# Full dataset — one prompt, strict hair/eye protection, explicit fallback
เปิด **Colab GPU แล้ว Run all** ได้จากต้นไฟล์ ใช้ภาพก่อน diffusion ที่มีอยู่ใน Drive ไม่ฝึกใหม่ ไม่สร้าง HairFastGAN/Barbershop ซ้ำ

ตั้ง SOURCE_RUN เป็นโฟลเดอร์ล่าสุดที่คุณส่งมาแล้ว `strict_hair_bg_20261009_071358_e864f4` หากย้ายให้แก้เฉพาะช่องนี้

- คืน `safety_checker=None` ให้ตรงกับ EyeModes ต้นฉบับสำหรับการทดลองภาพบุคคลนี้
- โหลด diffusion หนึ่งครั้ง ใช้หนึ่ง prompt/seed; ล็อก RGB ผมใน mask + ปกป้องตา; sharpen พื้นหลังแยก
- ตรวจตาไม่ได้ → **คืนภาพก่อน diffusion ทั้งภาพโดยไม่แก้** พร้อมสถานะ `fallback_original`; ไม่เดา ไม่ตัดคู่ข้อมูล
- เปรียบเทียบผลของระบบครบทุกคู่และรายงานอัตราสำเร็จควบคู่ ไม่อ้างว่าภาพ fallback ผ่าน diffusion
- ถ้าพื้นที่แก้ใบหน้าไม่เหลือ ใช้ `background_only` แยกอีกสถานะ
- คาดหวัง 285 before + 285 after สำหรับ 19 คู่ × 5 trials × 3 แขน; ตรวจจำนวนก่อนออกรายงาน

ผลใหม่อยู่ในโฟลเดอร์ `full_cases_...` ไม่เขียนทับผลเดิม หาก runtime หลุดให้ใส่ RESUME_OUTPUT เพื่อรันต่อ

ยังไม่สามารถรับประกันทุกข้อผิดพลาด เช่นไฟล์หาย พื้นที่เต็ม หรือ runtime หลุด แต่ไฟล์นี้ไม่ตัดภาพออกเพราะสองปัญหาที่รายงานมา


In [ ]:
from google.colab import drive,files
from pathlib import Path
import os,sys,json,hashlib,shutil,subprocess,time,uuid,gc,copy
import numpy as np,pandas as pd
from PIL import Image,ImageOps,ImageFilter,ImageDraw

drive.mount('/content/drive')
SOURCE_RUN='/content/drive/MyDrive/Rotation_Diffusion_EyeMask_Study/strict_hair_bg_20261009_071358_e864f4'
SOURCE_REPORT='' # ปกติหาได้จาก provenance ของรอบเดิม; กรอกเมื่อย้ายโฟลเดอร์
RESUME_OUTPUT='' # ว่าง=รอบใหม่; ใส่โฟลเดอร์ full_cases_... เพื่อรันต่อหลัง runtime หลุด
ROOT=Path('/content/drive/MyDrive/Rotation_Diffusion_EyeMask_Study')

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(8*1024**2),b''):h.update(part)
    return h.hexdigest()
def need(path):
    if not path.is_file():raise FileNotFoundError('ต้องใช้ไฟล์เดิม: '+str(path))
    return path
source=Path(SOURCE_RUN)
original_records=json.loads(need(source/'base_records.json').read_text())
ancestor=source;visited=set();old_hashes={}
while str(ancestor) not in visited:
    visited.add(str(ancestor))
    if (ancestor/'input_hashes.json').is_file():
        old_hashes=json.loads((ancestor/'input_hashes.json').read_text());break
    p=ancestor/'source_run.json'
    if not p.is_file():break
    ancestor=Path(json.loads(p.read_text())['previous_output'])
if SOURCE_REPORT:REPORT_ROOT=Path(SOURCE_REPORT)
else:
    candidates=set()
    for name in old_hashes:
        p=Path(name)
        if len(p.parents)>=4 and p.parent.parent.name=='images':
            root=p.parents[3]
            if (root/'common_protocol.json').is_file():candidates.add(root)
    assert len(candidates)==1,'หา common report ต้นทางไม่ชัดเจน: ใส่ SOURCE_REPORT ด้านบน'
    REPORT_ROOT=candidates.pop()
SOURCE_PROTOCOL=json.loads(need(REPORT_ROOT/'common_protocol.json').read_text())
SOURCE_LOCK=json.loads(need(REPORT_ROOT/'checkpoint_lock.json').read_text())
SPLITS=SOURCE_PROTOCOL['splits'];TRIALS=['Baseline','A','B','C','D']
BRANCHES=['HFG_rotate_on','HFG_rotate_off','Barbershop']
expected={(s,t,k,b) for s,keys in SPLITS.items() for t in TRIALS for k in keys for b in BRANCHES}
assert len(original_records)==len(expected)
assert {(r['split'],r['trial'],r['case'],r['branch']) for r in original_records}==expected
for r in original_records:assert sha(need(Path(r['path'])))==r['sha256']
INPUT_HASHES={}
for split,keys in SPLITS.items():
    for key in keys:
        for field in ['face','ref','shape','target','protected','eval_core','eval_refmask']:
            p=need(REPORT_ROOT/split/'images'/key/(field+'.png'))
            INPUT_HASHES[str(p)]=sha(p)
            if str(p) in old_hashes:assert INPUT_HASHES[str(p)]==old_hashes[str(p)]
CONFIG=dict(version='complete_cases_v1',source_run=str(source),source_report=str(REPORT_ROOT),
    source_base_records_sha256=sha(source/'base_records.json'),
    prompt='Photorealistic portrait, subtly refine existing facial skin texture. Preserve identity, facial geometry, expression, gaze, hair, and hair color. No makeup changes.',
    negative='different identity, altered eyes, changed hair color, changed hairstyle, makeup, cartoon, distorted face, artifacts',
    seed=3407,strength=.5,steps=20,guidance=7.5,face_blend=.35,
    hair_margin=5,sharpen_radius=2.,sharpen_percent=120,sharpen_threshold=3,
    safety_checker='None, matching the supplied EyeModes notebook',
    failure_policy='return pre-diffusion image unchanged; explicitly label fallback; never claim fallback is diffusion success')
if RESUME_OUTPUT:
    OUT=Path(RESUME_OUTPUT);assert json.loads(need(OUT/'run_config.json').read_text())==CONFIG
    BASE_RECORDS=json.loads(need(OUT/'base_records.json').read_text())
else:
    OUT=ROOT/('full_cases_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]);OUT.mkdir(parents=True)
    BASE_RECORDS=[]
    for r in original_records:
        folder=OUT/r['split']/r['trial']/r['case']/r['branch'];folder.mkdir(parents=True)
        dest=folder/'before.png';shutil.copy2(r['path'],dest)
        BASE_RECORDS.append(dict(r,path=str(dest),sha256=sha(dest)))
    (OUT/'base_records.json').write_text(json.dumps(BASE_RECORDS,indent=2))
    (OUT/'run_config.json').write_text(json.dumps(CONFIG,indent=2))
RESULTS_ROOT=OUT
NEED_EVALUATOR=not all(n in globals() for n in ['hair_fast','id_net','parser_net','shape_metrics','extra_metrics','quantile_color','read_mask','gpu'])
print('Output:',OUT,'| expected before:',len(BASE_RECORDS),'| expected after:',len(BASE_RECORDS))
print('Reuse images/checkpoints; no training and no rotation re-generation. One prompt, one seed.')


In [ ]:
SOURCE_NOTEBOOK_SHA256='27f2607ebb1bfb5d3a48221d0e097bc8877520f2e449a600f438095df8b05fc9'
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
if NEED_EVALUATOR:
    from google.colab import drive
    from pathlib import Path
    import os, shutil, subprocess, sys, json, time
    from huggingface_hub import snapshot_download
    
    
    LOCAL_REPO = '/content/HairFastGAN_rotation_eyes'
    DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
    
    # ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
    HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
    if not Path(LOCAL_REPO, '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
    if HAIRFAST_REVISION:
        subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
    os.chdir(LOCAL_REPO)
    local_models = Path(LOCAL_REPO, 'pretrained_models')
    if not local_models.exists() or not any(local_models.iterdir()):
        if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
            shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
        else:
            snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
            shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
    
    # ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
    for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
        p = local_models / relative
        if not p.exists() or p.stat().st_size < 1000000:
            raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
    Path('input').mkdir(exist_ok=True)
    Path('output').mkdir(exist_ok=True)
    REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    print('Repo commit:', REPO_COMMIT)


In [ ]:
if NEED_EVALUATOR:
    # Compatibility backend: native torch operators also execute on CUDA tensors.
    # No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
    import ast, hashlib, importlib, sys
    from pathlib import Path
    STYLEGAN_OPS_BACKEND = 'torch_native'
    
    def _native_stylegan_source(source, kind):
        marker = '# HAIRFAST_TORCH_NATIVE_V1'
        if marker in source:
            return source
        tree = ast.parse(source)
        extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
        entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
        edits = []
        found_load = found_entry = False
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
                if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                    raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
                edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
                found_load = True
            if isinstance(node, ast.FunctionDef) and node.name == entry:
                edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
                found_entry = True
        if not found_load or not found_entry:
            raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
        lines = source.splitlines(True)
        for start,end,text in sorted(edits, reverse=True):
            lines[start:end] = [text]
        updated = marker + '\n' + ''.join(lines)
        if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
            updated += '\n\n' + UP_NATIVE_IMPL + '\n'
        compile(updated, kind + '.py', 'exec')
        return updated
    
    FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
    UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
    UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'
    
    def install_native_stylegan_ops(repo):
        root = Path(repo).resolve()
        pairs = sorted((root / 'models').rglob('op/fused_act.py'))
        if not pairs:
            raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
        planned = []
        for fused_path in pairs:
            for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
                if not path.is_file():
                    raise RuntimeError('ขาด operator: ' + str(path))
                module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
                if module_name in sys.modules:
                    raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
                old = path.read_text()
                planned.append((path, old, _native_stylegan_source(old,kind)))
        # Validate every candidate before writing; preserve each original once.
        records = []
        for path,old,new in planned:
            if old != new:
                backup = path.with_suffix('.py.before_native')
                if not backup.exists(): backup.write_text(old)
                path.write_text(new)
            records.append({'path':str(path.relative_to(root)),
                            'sha256':hashlib.sha256(new.encode()).hexdigest()})
        importlib.invalidate_caches()
        return records
    
    STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
    print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))
    


In [ ]:
if NEED_EVALUATOR:
    from PIL import Image
    if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
    import sys, torch
    sys.path.insert(0, LOCAL_REPO)
    from hair_swap import HairFast, get_parser
    hair_fast = HairFast(get_parser().parse_args([]))
    print('HairFastGAN พร้อม')
    
    import numpy as np,pandas as pd
    from PIL import Image,ImageOps,ImageFilter,ImageDraw
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint
    from models.Net import get_segmentation as segment
    from IPython.display import display
    b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
    for module in [enc,G,pp,align]:module.eval().requires_grad_(False)


In [ ]:
if NEED_EVALUATOR:
    runtime_baseS={}
    def cpu(x):
        if torch.is_tensor(x):
            return x.detach().cpu().clone()
        if isinstance(x, dict):
            return {k: cpu(v) for k, v in x.items()}
        if isinstance(x, list):
            return [cpu(v) for v in x]
        if isinstance(x, tuple):
            return tuple((cpu(v) for v in x))
        return copy.deepcopy(x)
    
    def gpu(x):
        if torch.is_tensor(x):
            return x.to('cuda').clone()
        if isinstance(x, dict):
            return {k: gpu(v) for k, v in x.items()}
        if isinstance(x, list):return [gpu(v) for v in x]
        if isinstance(x, tuple):return tuple(gpu(v) for v in x)
        return x
    
    def resize(x):
        return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
    
    def saveim(x, p):
        a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
        Image.fromarray((a * 255).round().astype('uint8')).save(p)
    
    def lab(x):
        x = x.float().clamp(0, 1)
        lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
        M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
        xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
        f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
        return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)
    
    def render(s, d):
        S = torch.cat([d['S'][:, :6], s], 1)
        return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]
    
    def finish(raw, d):
        sf, ff = pp(d['I1'], b.downsample_256(raw))
        return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]
    
    def detail(x):
        y = x.mean(1, keepdim=True)
        mu = F.avg_pool2d(y, 9, 1, 4)
        v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
        return (y - mu) / (v + 0.001).sqrt()
    
    def unit(raw):return resize(((raw+1)/2).clamp(0,1))
    def read_mask(p):
        with Image.open(p) as im:
            if im.size!=(256,256):raise ValueError('Mask size mismatch')
            a=np.asarray(im.convert('L'))>127
        return torch.from_numpy(a.copy())[None,None]
    
    def load_case(k):
        d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
        folder=STUDY_ROOT/'trial_A/inputs'/k
        for name in ['eval_core','eval_refmask','target','protected']:
            mask=read_mask(folder/(name+'.png'))
            if name in ['target','protected']:
                assert torch.equal(d[name].bool(),mask), (k,name)
            d[name]=mask
        for name in ['core','refmask']:d[name]=d[name].bool()
        for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
        return d
    
    def quantile_color(im,ref,mask,refmask):
        l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
        q=torch.linspace(.1,.9,9,device=im.device)
        diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
        return diff[:,0].mean(),diff[:,1:].mean()
    
    def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))
    
    def shape_metrics(im,d):
        labels=segment(align.to_bisenet(im))
        pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
        target=d['target']
        return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                    target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))
    
    
    # Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
    from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
    from losses.pp_losses import Backbone
    from huggingface_hub import hf_hub_download
    id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
    if not id_path.is_file():
        downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
        id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
    id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
    id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
    id_net=id_net.cuda().eval().requires_grad_(False)
    FaceParsing_tensor.parsing_img()
    parser_net=FaceParsing.bise_net
    if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
    parser_net.eval().requires_grad_(False)
    assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13
    
    def id_features(im):
        x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
        x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
        feats=id_net(x)
        return F.normalize(feats,dim=1)
    
    def constraint_terms(im,d,fx):
        # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
        logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
        other=torch.cat((logits[:,:17],logits[:,18:]),1)
        binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
        target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
        if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
        prob=binary.sigmoid()
        bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
        balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
        dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
        embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
        identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
        return identity,balanced+dice,balanced,dice
    
    def to_size(x,size):
        return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)
    
    def detail_terms(full,d,fx):
        values={}
        for size in [256,512]:
            x=to_size(full,size)
            teacher=fx['pre'] if size==256 else fx['pre_512']
            mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
            inner=-F.max_pool2d(-mask,9,1,4)
            if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
            values[size]=masked_mae(detail(x),detail(teacher),inner)
        return values
    
    
    from scipy.ndimage import binary_erosion,binary_dilation
    from skimage.metrics import structural_similarity
    _basic_shape_metrics=shape_metrics
    def shape_metrics(im,d):
        result=_basic_shape_metrics(im,d)
        labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
        t=d['target'][0,0].cpu().numpy()>.5
        result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
        a=p^binary_erosion(p);bnd=t^binary_erosion(t)
        prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
        rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
        result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
        return result
    
    def extra_metrics(im,d):
        x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
        mask=d['protected'][0,0].cpu().numpy()>.5
        _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
        safe=binary_erosion(mask,iterations=3)
        if not safe.any():raise ValueError('No protected SSIM interior')
        mse=float(((x-y)**2)[mask].mean())
        return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
            any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


In [ ]:
# ข้อความฐานและ negative เหมือนกันทุก prompt
BASE_PROMPT = (
    'Photorealistic portrait, refine the existing hair in the input image, '
    'preserve the current hairstyle silhouette, length and hair color, '
    'preserve facial features and the surrounding scene.'
)
COMMON_NEGATIVE = (
    'cartoon, anime, CGI, wig, plastic hair, deformed face, altered facial features, '
    'different haircut, changed hair color, artificial halo, artifacts'
)
FOCUS_PHRASES = {
    'prompt1_texture': 'Natural hair texture with subtle strand variation, avoid rigid clumps and waxy surfaces.',
    'prompt2_color': 'Preserve the existing hair hue and saturation, retain natural tonal variation.',
    'prompt3_sharpness': 'Fine individual hair strands and realistic strand separation without harsh oversharpening.',
    'prompt4_blend': 'Natural soft hairline transition and seamless hair boundary without a pasted-on edge.',
    'prompt5_lighting': 'Hair highlights and shadows consistent with the existing facial illumination.',
}
REFINE_PROMPTS = [
    {'tag': tag, 'prompt': BASE_PROMPT + ' ' + phrase, 'negative_prompt': COMMON_NEGATIVE}
    for tag, phrase in FOCUS_PHRASES.items()
]
REFINE_PROMPTS.append({'tag':'prompt6_combined',
    'prompt': BASE_PROMPT + ' ' + ' '.join(FOCUS_PHRASES.values()),
    'negative_prompt':COMMON_NEGATIVE})
# Reference เข้าขั้น HairFastGAN เท่านั้น; diffusion นี้เห็น A ไม่ได้เห็น reference โดยตรง
DEFAULT_PROMPT_TAG = 'prompt6_combined'
INPAINT_STEPS = 20
INPAINT_STRENGTH = 0.5  # จุดเริ่มทดลอง ไม่ใช่ค่าที่รับรองว่าดีที่สุด
INPAINT_GUIDANCE = 7.5
DEFAULT_SEED = 3407
HAIRFAST_SEED = 3407
FACE_MARGIN_PX = 3      # pixels ที่ resolution ของภาพ A
BLUR_RADIUS = 2.0
# Fixed value range, ไม่ normalize ด้วย min/max ต่อภาพเพราะจะทำให้สีเปลี่ยน
# official HairFast output ใช้ภาพในช่วง [0,1]; ถ้า repo ที่ใช้ต่างไปให้ตรวจและเปลี่ยน
OUTPUT_TENSOR_RANGE = 'zero_one'
PROMPTS = {p['tag']: p for p in REFINE_PROMPTS}


def segment_labels(image):
    inputs = seg_processor(images=image.convert('RGB'), return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        logits = seg_model(**inputs).logits
        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode='bilinear', align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)


def build_masks(labels, margin_px=FACE_MARGIN_PX, blur_radius=BLUR_RADIUS):
    hair = labels == HAIR_ID
    face = labels == FACE_ID
    if not hair.any():
        raise ValueError('ไม่พบเส้นผมในภาพ กรุณาตรวจภาพหรือ segmentation')
    if not face.any():
        raise ValueError('ไม่พบใบหน้า ไม่สามารถใช้โหมดปกป้องหน้าได้')
    face_img = Image.fromarray(face.astype(np.uint8) * 255)
    protected = np.asarray(face_img.filter(ImageFilter.MaxFilter(2 * margin_px + 1))) > 0
    hair_img = Image.fromarray(hair.astype(np.uint8) * 255)
    soft = np.asarray(hair_img.filter(ImageFilter.GaussianBlur(blur_radius)), dtype=np.float32) / 255
    # C แก้เฉพาะผมที่ segmentation เลือก ไม่แก้ background/face แม้ขอบ blur ล้ำ
    allowed = hair & ~protected
    alpha = soft * allowed
    if not np.any(alpha > 0):
        raise ValueError('พื้นที่ผมหลังกันหน้าว่าง ต้องตรวจ mask')
    return {'hair': hair, 'face': face, 'protected': protected,
            'alpha': alpha.astype(np.float32), 'raw_mask': Image.fromarray(np.rint(soft * 255).astype(np.uint8))}


def composite_protected(base, edited, alpha):
    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):
        raise ValueError('ภาพและ alpha ต้องมีขนาดตรงกัน')
    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]
    original = np.array(base.convert('RGB'))
    changed = np.asarray(edited.convert('RGB'), dtype=np.float32)
    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)
    output[a[..., 0] == 0] = original[a[..., 0] == 0]
    return Image.fromarray(output)


def mask_overlay(image, masks):
    a = np.asarray(image.convert('RGB')).copy().astype(float)
    for mask, rgb in [(masks['alpha'] > 0, (0, 255, 0)), (masks['protected'], (255, 0, 0))]:
        a[mask] = 0.55 * a[mask] + 0.45 * np.array(rgb)
    return Image.fromarray(a.astype(np.uint8))


def padded(image, target=512, mask=False):
    scale = target / max(image.size)
    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)
    canvas = Image.new(image.mode, (target, target), 0)
    x, y = (target-size[0])//2, (target-size[1])//2
    canvas.paste(resized, (x, y))
    return canvas, (x, y, x+size[0], y+size[1])


def tensor_to_pil(value, value_range='zero_one'):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if torch.is_tensor(value):
        t = value.detach().cpu()
        if t.ndim == 4 and t.shape[0] == 1:
            t = t[0]
        if t.ndim != 3 or t.shape[0] != 3:
            raise ValueError(f'Unexpected tensor shape: {tuple(t.shape)}')
        a = t.permute(1, 2, 0).numpy()
    else:
        a = np.asarray(value)
    if a.dtype == np.uint8:
        return Image.fromarray(a).convert('RGB')
    if value_range == 'minus_one_one':
        a = (a + 1) / 2
    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:
        raise ValueError('Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ')
    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert('RGB')


def resolve_prompt_configs(overrides=None):
    # คัดลอกต่อคำขอ ไม่เปลี่ยน preset ส่วนกลาง
    configs = {tag:dict(cfg) for tag,cfg in PROMPTS.items()}
    if overrides is None:
        return configs
    if not isinstance(overrides, dict) or len(overrides) > len(PROMPTS):
        raise ValueError('prompt_overrides ต้องเป็น object ตามชื่อ prompt ที่รองรับ')
    for tag, change in overrides.items():
        if tag not in configs or not isinstance(change, dict):
            raise ValueError('ชื่อ prompt หรือรูปแบบ override ไม่ถูกต้อง')
        if set(change) - {'prompt','negative_prompt'}:
            raise ValueError('แก้ได้เฉพาะ prompt และ negative_prompt')
        for field, text in change.items():
            if not isinstance(text, str) or len(text) > 4000:
                raise ValueError('ข้อความ prompt ต้องเป็น string ไม่เกิน 4000 ตัวอักษร')
            if field == 'prompt' and not text.strip():
                raise ValueError('prompt ห้ามว่าง')
            configs[tag][field] = text
    return configs


def parse_prompt_changes(prompt_tag, mode, prompt=None, negative_prompt=None, prompt_overrides=None):
    import json
    if prompt_tag not in PROMPTS:
        raise ValueError('ไม่พบ prompt_tag')
    if prompt_overrides is not None:
        if not isinstance(prompt_overrides, str) or len(prompt_overrides) > 60000:
            raise ValueError('prompt_overrides ต้องเป็น JSON string ไม่เกิน 60000 ตัวอักษร')
        try:
            changes = json.loads(prompt_overrides)
        except (ValueError, TypeError):
            raise ValueError('prompt_overrides เป็น JSON ไม่ถูกต้อง')
        resolve_prompt_configs(changes)
        if not isinstance(changes, dict):
            raise ValueError('prompt_overrides ต้องเป็น JSON object')
    else:
        changes = {}
    if prompt is not None or negative_prompt is not None:
        if mode in ('all_prompts', 'all_prompts_unprotected', 'all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine'):
            raise ValueError('all_prompts ให้ใช้ prompt_overrides เพื่อระบุข้อความแยกแต่ละชุด')
        if prompt_tag in changes:
            raise ValueError('ส่งข้อความซ้ำกัน: เลือกใช้ prompt/negative_prompt หรือ prompt_overrides')
        changes[prompt_tag] = {}
        if prompt is not None:
            changes[prompt_tag]['prompt'] = prompt
        if negative_prompt is not None:
            changes[prompt_tag]['negative_prompt'] = negative_prompt
    if mode == 'baseline' and changes:
        raise ValueError('baseline ไม่ใช้ diffusion prompt ให้เลือก protected/abc/all_prompts')
    if mode not in ('all_prompts', 'all_prompts_unprotected', 'all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine') and set(changes) - {prompt_tag}:
        raise ValueError('โหมดนี้แก้ได้เฉพาะ prompt_tag ที่เลือก')
    resolve_prompt_configs(changes)
    return changes


def resolve_inpaint_settings(strength=None, steps=None, guidance=None):
    import math
    strength = float(INPAINT_STRENGTH if strength is None else strength)
    steps = INPAINT_STEPS if steps is None else steps
    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)
    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:
        raise ValueError('steps ต้องเป็นจำนวนเต็ม 1–100')
    steps = int(steps)
    if not math.isfinite(strength) or not 0 < strength <= 1:
        raise ValueError('strength ต้องมากกว่า 0 และไม่เกิน 1')
    if not math.isfinite(guidance) or not 1 <= guidance <= 20:
        raise ValueError('guidance ต้องอยู่ในช่วง 1–20')
    if int(steps * strength) < 1:
        raise ValueError('steps × strength ต้องไม่น้อยกว่า 1')
    return {'strength':strength, 'steps':steps, 'guidance':guidance}


def refine_once(
    base, masks, prompt_config, seed, protected,
    strength=None, steps=None, guidance=None
):
    settings = resolve_inpaint_settings(strength, steps, guidance)
    image512, box = padded(base)

    if protected:
        alpha_img = Image.fromarray(
            np.rint(masks['alpha'] * 255).astype(np.uint8)
        )
        mask512, _ = padded(alpha_img, mask=True)

        protect512, _ = padded(
            Image.fromarray(
                masks['protected'].astype(np.uint8) * 255
            ),
            mask=True
        )
        protect512 = np.asarray(
            protect512.filter(ImageFilter.MaxFilter(3))
        ) > 0

        m = np.array(mask512)
        m[protect512] = 0
        mask512 = Image.fromarray(m)
    else:
        mask512, _ = padded(masks['raw_mask'], mask=True)

    if not (np.asarray(mask512) > 127).any():
        raise ValueError('mask เล็กเกินไปหลัง resize เป็น 512')

    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    with torch.inference_mode():
        result = inpaint_pipe(
            image=image512,
            mask_image=mask512,
            prompt=prompt_config['prompt'],
            negative_prompt=prompt_config['negative_prompt'],
            num_inference_steps=settings["steps"],
            strength=settings["strength"],
            guidance_scale=settings["guidance"],
            generator=generator
        )

    flags = getattr(result, 'nsfw_content_detected', None)
    if flags is not None and any(flags):
        raise ValueError('ผลภาพถูกตัวกรองของโมเดลระงับ')

    edited = (
        result.images[0]
        .convert('RGB')
        .crop(box)
        .resize(base.size, Image.Resampling.LANCZOS)
    )

    return (
        composite_protected(base, edited, masks['alpha'])
        if protected
        else edited
    )


# Single-pass joint hair + face study: same six prompts/noise/mask in both modes.
JOINT_MODES = ('all_prompts_joint_controlled','all_prompts_joint_unprotected','all_prompts_joint_controlled_noeyes','all_prompts_joint_unprotected_noeyes','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')
JOINT_FACE_BLEND = 0.35
JOINT_TRANSITION_PX = 24  # measured at 512px short side; scaled with image size
JOINT_MARGIN_PX = 12
JOINT_INSTRUCTION = (
    'Photorealistic portrait of the same person, consistent hair and facial lighting, seamless hairline. '
    'Preserve facial features, expression, hairstyle and hair color. '
)
JOINT_FOCUS_PHRASES = {**FOCUS_PHRASES,
    'prompt6_combined': 'Natural strands, accurate hair color, fine detail without oversharpening, soft hairline, realistic highlights and shadows.'}



JOINT_CONTROLLED_MODES = ('all_prompts_joint_controlled', 'all_prompts_joint_controlled_noeyes', 'all_prompts_joint_controlled_eyerefine')
JOINT_EYE_MODES = ('all_prompts_joint_controlled', 'all_prompts_joint_unprotected','all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')

def resolve_joint_settings(face_blend=JOINT_FACE_BLEND):
    import math
    face_blend = float(face_blend)
    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:
        raise ValueError('face_blend ต้องอยู่ในช่วง 0–1')
    return {'face_blend':face_blend}


def joint_prompt_configs(configs):
    # Keep P1–P6 axes; replace the hair-only preamble with a short joint preamble.
    # Compact P6 to avoid swallowing its five axes in a long prompt.
    result = {}
    for tag,cfg in configs.items():
        body = JOINT_FOCUS_PHRASES[tag] if cfg['prompt'] == PROMPTS[tag]['prompt'] else cfg['prompt']
        result[tag] = {**cfg, 'prompt':JOINT_INSTRUCTION + body}
    return result


def validate_joint_prompt_lengths(configs):
    # Validate using the tokenizer actually loaded in Colab, not character counts.
    tokenizer = getattr(inpaint_pipe, 'tokenizer', None)
    if tokenizer is None:
        return
    limit = tokenizer.model_max_length
    for tag,cfg in configs.items():
        for field in ('prompt','negative_prompt'):
            count = len(tokenizer(cfg[field], truncation=False)['input_ids'])
            if count > limit:
                raise ValueError(f'{tag} {field}: {count} tokens เกิน {limit}; ย่อข้อความเพื่อไม่ให้ส่วนท้ายถูกตัด')


# รักษาตาจากภาพ A ใน Joint ทั้งสองโหมด (ปิดได้เพื่อเทียบผลเดิม)
# Eye protection เลือกตาม mode ของแต่ละคำขอ ไม่ใช้ global switch
JOINT_EYE_MARGIN_PX = 3     # ที่ด้านสั้น 512: ครอบขอบเปลือกตา
JOINT_EYE_FEATHER_PX = 6    # ไล่ขอบออกด้านนอก ไม่ลดการปกป้องแกนกลาง
_EYE_LANDMARK_MODEL = None


def detect_eye_landmarks(base):
    """ใช้ face-alignment ที่ติดตั้งไว้แล้ว; โหลดครั้งแรก ใช้ CPU ลด VRAM."""
    global _EYE_LANDMARK_MODEL
    import face_alignment
    import inspect
    if _EYE_LANDMARK_MODEL is None:
        kwargs = dict(device='cpu', flip_input=False)
        if 'compile' in inspect.signature(face_alignment.FaceAlignment).parameters:
            kwargs['compile'] = False
        _EYE_LANDMARK_MODEL = face_alignment.FaceAlignment(
            face_alignment.LandmarksType.TWO_D, **kwargs)
    with torch.inference_mode():
        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert('RGB')))
    if predictions is None or len(predictions) != 1:
        raise ValueError('Eye protection: ต้องตรวจพบใบหน้าเดียวในภาพ A กรุณาใช้ภาพหน้าชัดเจน')
    points = np.asarray(predictions[0], dtype=np.float32)
    if points.shape != (68, 2) or not np.isfinite(points).all():
        raise ValueError('Eye protection: landmarks ไม่ถูกต้อง')
    return points


def build_eye_protection(size, landmarks):
    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""
    from scipy.ndimage import distance_transform_edt
    w, h = size
    canvas = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(canvas)
    for eye in (landmarks[36:42], landmarks[42:48]):
        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():
            raise ValueError('Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป')
        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)
    scale = min(w, h) / 512.0
    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))
    if margin:
        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))
    core = np.asarray(canvas) > 0
    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)
    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)
    keep = (1 - x*x*(3-2*x)).astype(np.float32)
    keep[core] = 1.0
    return {'eye_core': core, 'eye_keep': keep}


def build_joint_masks(masks, face_blend=JOINT_FACE_BLEND, base=None, mode='all_prompts_joint_controlled'):
    if mode not in JOINT_MODES:
        raise ValueError('ไม่ใช่โหมด joint')
    from scipy.ndimage import binary_fill_holes, distance_transform_edt
    face_blend = resolve_joint_settings(face_blend)['face_blend']
    h,w = masks['face'].shape
    scale = min(h,w) / 512.0
    margin = max(1, round(JOINT_MARGIN_PX * scale))
    transition = max(1.0, JOINT_TRANSITION_PX * scale)
    # Fill enclosed holes (e.g. glasses) and close small segmentation gaps at hairline.
    union = masks['hair'] | masks['face']
    radius = max(1, round(3 * scale))
    u = Image.fromarray(union.astype(np.uint8)*255)
    closed = np.asarray(u.filter(ImageFilter.MaxFilter(2*radius+1)).filter(
        ImageFilter.MinFilter(2*radius+1))) > 0
    core = binary_fill_holes(union | closed)
    core_img = Image.fromarray(core.astype(np.uint8)*255)
    support = np.asarray(core_img.filter(ImageFilter.MaxFilter(2*margin+1))) > 0
    # Feather only the outer perimeter; never cut the shared face/hair boundary.
    distance = distance_transform_edt(np.pad(support, 1))[1:-1,1:-1]
    x = np.clip(distance / margin, 0, 1)
    outer_alpha = (x*x*(3-2*x)).astype(np.float32)
    outer_alpha[core] = 1.0
    # Smoothly reduce generated-image weight towards the interior of the face.
    face = binary_fill_holes(masks['face']) & ~masks['hair']
    distance_face = distance_transform_edt(np.pad(face, 1))[1:-1,1:-1]
    x = np.clip(distance_face / transition, 0, 1)
    face_weight = (x*x*(3-2*x)).astype(np.float32)
    controlled_alpha = outer_alpha * (1 - (1-face_blend)*face_weight)
    if mode in JOINT_EYE_MODES:
        if base is not None:
            if base.size != (w, h):
                raise ValueError('Eye protection: ภาพ A และ mask ขนาดไม่ตรงกัน')
            masks.update(build_eye_protection(base.size, detect_eye_landmarks(base)))
        if 'eye_keep' not in masks:
            raise ValueError('Eye protection: ต้องส่งภาพ A เพื่อหาตำแหน่งตาก่อน')
        eye_keep, eye_core = masks['eye_keep'], masks['eye_core']
    else:
        eye_keep = np.zeros((h, w), dtype=np.float32)
        eye_core = np.zeros((h, w), dtype=bool)
    return {'eye_keep':eye_keep, 'eye_core':eye_core,
            'raw_mask':Image.fromarray(support.astype(np.uint8)*255),
            'core':core, 'support':support, 'face_weight':face_weight,
            'outer_alpha':outer_alpha, 'controlled_alpha':controlled_alpha,
            'margin_px_actual':margin, 'transition_px_actual':transition}


def joint_alpha(joint, mode):
    if mode not in JOINT_MODES:
        raise ValueError('ไม่ใช่โหมด joint')
    alpha = joint['controlled_alpha'] if mode in JOINT_CONTROLLED_MODES else joint['outer_alpha']
    return alpha * (1 - joint['eye_keep']) if mode in JOINT_EYE_MODES else alpha.copy()



JOINT_EYE_REFINE_MODES = ('all_prompts_joint_controlled_eyerefine','all_prompts_joint_unprotected_eyerefine')
EYE_REFINE_STRENGTH = 0.20  # ความแรงการสร้างรายละเอียดตา แยกจาก Strength ของ Joint
EYE_REFINE_BLEND = 0.35     # น้ำหนักตาที่ AI ปรับ; ที่เหลือใช้ A
EYE_REFINE_STEPS = 30
EYE_REFINE_GUIDANCE = 4.0
EYE_REFINE_PROMPT = ('Photorealistic eyes, subtly refine existing eye detail. Preserve eye shape, '
    'iris size, iris color, gaze direction and eyelids. Natural texture and soft lighting.')
EYE_REFINE_NEGATIVE = ('different eye shape, different iris color, changed gaze, enlarged eyes, '
    'asymmetric pupils, extra pupils, heavy makeup, oversharpening, artifacts')


def refine_eyes_from_a(base, joint, seed):
    # Cache ต่อคำขอ: ทั้ง P1–P6 ใช้ตาปรับชุดเดียวกัน ไม่สร้างหกรอบ
    import hashlib
    key=(hashlib.sha256(base.tobytes()).hexdigest(), int(seed), EYE_REFINE_STRENGTH,
         EYE_REFINE_BLEND,EYE_REFINE_STEPS,EYE_REFINE_GUIDANCE,
         EYE_REFINE_PROMPT,EYE_REFINE_NEGATIVE)
    if joint.get('_eye_cache_key')==key:
        return joint['_eye_cache_image']
    if not 0<=EYE_REFINE_BLEND<=1:
        raise ValueError('EYE_REFINE_BLEND ต้องอยู่ในช่วง 0–1')
    if EYE_REFINE_BLEND==0:
        return base.copy()
    ys,xs=np.where(joint['eye_keep']>0)
    if len(xs)==0: raise ValueError('ไม่พบพื้นที่ตาสำหรับปรับรายละเอียด')
    # ครอปตาคู่พร้อมบริบทจาก A เพื่อไม่ย่อทั้งหน้าแล้วสูญรายละเอียดตา
    width=xs.max()-xs.min()+1; height=ys.max()-ys.min()+1
    padx=max(16,round(width*.25));pady=max(16,round(height*1.5))
    box=(max(0,int(xs.min()-padx)),max(0,int(ys.min()-pady)),
         min(base.width,int(xs.max()+1+padx)),min(base.height,int(ys.max()+1+pady)))
    x0,y0,x1,y1=box
    crop=base.crop(box)
    raw_mask=Image.fromarray((joint['eye_keep'][y0:y1,x0:x1]>0).astype(np.uint8)*255)
    cfg={'tag':'eye_detail_from_A','prompt':EYE_REFINE_PROMPT,'negative_prompt':EYE_REFINE_NEGATIVE}
    validate_joint_prompt_lengths({'eye_detail_from_A':cfg})
    edited=refine_once(crop,{'raw_mask':raw_mask},cfg,seed,False,
        strength=EYE_REFINE_STRENGTH,steps=EYE_REFINE_STEPS,guidance=EYE_REFINE_GUIDANCE)
    canvas=base.copy();canvas.paste(edited,box[:2])
    # น้ำหนักคงที่ก่อนประกอบด้วย eye_keep อีกครั้ง: ป้องกัน feather ซ้ำ
    candidate=composite_protected(base,canvas,np.full((base.height,base.width),EYE_REFINE_BLEND,dtype=np.float32))
    joint['_eye_cache_key']=key;joint['_eye_cache_image']=candidate
    return candidate

def refine_joint(base, masks, joint, prompt_config, seed, mode, **settings):
    # Exactly one diffusion call; both modes generate with the same full joint mask.
    inference_masks = {**masks, 'raw_mask':joint['raw_mask']}
    generated = refine_once(base, inference_masks, prompt_config, seed, False, **settings)
    # Eye modes คืนตาจาก A; noeyes ไม่มีการตัดตาออกจากน้ำหนัก Blend
    result = composite_protected(base, generated, joint_alpha(joint, mode))
    if mode in JOINT_EYE_REFINE_MODES:
        eye_image = refine_eyes_from_a(base, joint, seed)
        # เริ่มจากน้ำหนัก Joint ปกติ เพื่อไม่ feather ตา A ซ้ำ
        normal_alpha = joint['controlled_alpha'] if mode in JOINT_CONTROLLED_MODES else joint['outer_alpha']
        result = composite_protected(base, generated, normal_alpha)
        result = composite_protected(result, eye_image, joint['eye_keep'])
    return result


def joint_metadata(mode, face_blend):
    if mode not in JOINT_MODES:
        return {}
    blend = resolve_joint_settings(face_blend)['face_blend']
    return {'joint_refinement': {
        'eye_protection':mode in JOINT_EYE_MODES and mode not in JOINT_EYE_REFINE_MODES,
        'eye_refinement':mode in JOINT_EYE_REFINE_MODES,
        'eye_refine_strength':EYE_REFINE_STRENGTH if mode in JOINT_EYE_REFINE_MODES else None,
        'eye_refine_blend':EYE_REFINE_BLEND if mode in JOINT_EYE_REFINE_MODES else None,
        'extra_eye_passes_per_request':1 if mode in JOINT_EYE_REFINE_MODES and EYE_REFINE_BLEND>0 else 0,
        'eye_source':'HairFastGAN A' if mode in JOINT_EYE_MODES else None,
        'eye_margin_px_at_512':JOINT_EYE_MARGIN_PX,
        'eye_feather_px_at_512':JOINT_EYE_FEATHER_PX,
        'passes_per_prompt':1, 'face_control':'soft_blend' if mode in JOINT_CONTROLLED_MODES else 'none',
        'effective_face_blend':blend if mode in JOINT_CONTROLLED_MODES else 1.0,
        'face_blend_applied':mode in JOINT_CONTROLLED_MODES,
        'transition_px_at_512':JOINT_TRANSITION_PX, 'margin_px_at_512':JOINT_MARGIN_PX,
        'prompt_prefix':JOINT_INSTRUCTION,
        'mask_policy':'hair+face union, close small gaps, fill holes, expand outer perimeter',
        'outside_policy':'restore outside joint region with outer feather',
        'identity_conditioning':False, 'identity_guaranteed':False,
        'reference':'HairFastGAN output; no original-face identity conditioning',
    }}


## สร้างผลครบทุกคู่และบันทึกสถานะจริง

In [ ]:
from diffusers import StableDiffusionInpaintPipeline,UniPCMultistepScheduler
from transformers import SegformerImageProcessor,SegformerForSemanticSegmentation
from huggingface_hub import HfApi
import cv2
DEVICE='cuda'
# Free evaluator memory while diffusion runs; keep objects for the later metric phase.
for module in [enc,pp,G,align,hair_fast.embed.encoder,hair_fast.embed.e4e,id_net,parser_net]:module.cpu()
gc.collect();torch.cuda.empty_cache()
if 'seg_model' not in globals():
    seg_revision=HfApi().model_info('mattmdjaga/segformer_b2_clothes').sha
    seg_processor=SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes',revision=seg_revision)
    seg_model=SegformerForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes',revision=seg_revision)
seg_model.to(DEVICE).eval()
label_ids={str(v).lower():int(k) for k,v in seg_model.config.id2label.items()}
HAIR_ID,FACE_ID=label_ids['hair'],label_ids['face'];assert (HAIR_ID,FACE_ID)==(2,11)
if 'inpaint_pipe' not in globals():
    diff_revision=HfApi().model_info('stablediffusionapi/realistic-vision-v51').sha
    inpaint_pipe=StableDiffusionInpaintPipeline.from_pretrained('stablediffusionapi/realistic-vision-v51',revision=diff_revision,torch_dtype=torch.float16,safety_checker=None)
    inpaint_pipe.scheduler=UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)
    inpaint_pipe.enable_model_cpu_offload()
else:
    diff_revision=getattr(inpaint_pipe.config,'_commit_hash',None)
    if getattr(inpaint_pipe,'_execution_device',torch.device('cpu')).type=='cpu':inpaint_pipe.enable_model_cpu_offload()
inpaint_pipe.safety_checker=None # Restore explicit original notebook setting for this portrait experiment.
inpaint_pipe.set_progress_bar_config(disable=True)
if hasattr(inpaint_pipe.vae,'enable_slicing'):inpaint_pipe.vae.enable_slicing()
configs={'strict_hair_lock':dict(tag='strict_hair_lock',prompt=CONFIG['prompt'],negative_prompt=CONFIG['negative'])}
validate_joint_prompt_lengths(configs)
settings={k:CONFIG[k] for k in ['strength','steps','guidance']}
ENVIRONMENT=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
    diffusion_revision=diff_revision,seg_revision=getattr(seg_model.config,'_commit_hash',None),safety_checker=None)
(OUT/'environment.json').write_text(json.dumps(ENVIRONMENT,indent=2))
(OUT/'requirements.freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))

def atomic_json(path,data):
    tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2));os.replace(tmp,path)

def key_of(r):return (r['split'],r['trial'],r['case'],r['branch'])
ledger=OUT/'outcomes.json'
OUTCOMES=json.loads(ledger.read_text()) if ledger.exists() else []
done={key_of(r):r for r in OUTCOMES};assert len(done)==len(OUTCOMES)
for r in OUTCOMES:assert sha(need(Path(r['path'])))==r['sha256']

for rec in BASE_RECORDS:
    if key_of(rec) in done:continue
    before_path=Path(rec['path']);assert sha(before_path)==rec['sha256']
    with Image.open(before_path) as im:base=im.convert('RGB').copy()
    original=np.asarray(base);folder=before_path.parent
    status='diffusion_success';reason='';seconds=time.perf_counter()
    # A fallback is exactly the original image, never a guessed face/eye mask.
    eye=np.zeros((base.height,base.width),bool);hair=np.ones_like(eye);alpha=np.zeros_like(eye,dtype=np.float32)
    after=base.copy()
    try:
        labels=segment_labels(base);masks=build_masks(labels)
        if not masks['hair'].any():raise ValueError('No hair mask: strict color preservation cannot be checked')
        joint=build_joint_masks(masks,CONFIG['face_blend'],base=base,mode='all_prompts_joint_controlled')
        eye=joint['eye_core']
        with Image.open(REPORT_ROOT/rec['split']/'images'/rec['case']/'target.png') as im:
            target=np.asarray(im.convert('L').resize(base.size,Image.Resampling.NEAREST))>127
        hair=np.asarray(Image.fromarray((masks['hair']|target).astype('uint8')*255).filter(ImageFilter.MaxFilter(2*CONFIG['hair_margin']+1)))>0
        face_alpha=joint_alpha(joint,'all_prompts_joint_controlled');face_alpha[hair]=0
        raw=np.array(joint['raw_mask']);raw[hair|eye]=0
        bg=(labels==label_ids['background'])&~hair&~eye&~joint['support']
        inner=Image.fromarray(bg.astype('uint8')*255).filter(ImageFilter.MinFilter(5))
        bg_alpha=np.asarray(inner.filter(ImageFilter.GaussianBlur(2)),dtype=np.float32)/255.;bg_alpha[~bg]=0
        alpha=np.maximum(face_alpha,bg_alpha)
        if not (np.asarray(padded(Image.fromarray(raw),mask=True)[0])>127).any():
            status='background_only';reason='No editable face pixels remain after hair/eye protection';generated=base.copy()
        else:
            generated=refine_once(base,{**masks,'raw_mask':Image.fromarray(raw)},configs['strict_hair_lock'],CONFIG['seed'],False,**settings)
            if np.asarray(generated).max()==0:raise ValueError('Diffusion returned an all-black image')
        after=composite_protected(base,generated,face_alpha)
        sharp=base.filter(ImageFilter.UnsharpMask(radius=CONFIG['sharpen_radius'],percent=CONFIG['sharpen_percent'],threshold=CONFIG['sharpen_threshold']))
        after=composite_protected(after,sharp,bg_alpha)
        out=np.array(after);out[hair|eye]=original[hair|eye];after=Image.fromarray(out)
    except ValueError as error:
        known=('Eye protection:','No hair mask:','Diffusion returned an all-black image','ผลภาพถูกตัวกรองของโมเดลระงับ','mask เล็กเกินไปหลัง resize')
        if not str(error).startswith(known):raise
        status='fallback_original';reason=str(error);after=base.copy();alpha=np.zeros_like(alpha)
    except torch.OutOfMemoryError as error:
        status='fallback_original';reason='CUDA out of memory during postprocessing';after=base.copy();alpha=np.zeros_like(alpha)
        gc.collect();torch.cuda.empty_cache()
    arr=np.asarray(after)
    assert np.array_equal(arr[hair],original[hair]) and np.array_equal(arr[eye],original[eye])
    assert np.array_equal(arr[alpha==0],original[alpha==0])
    if status=='fallback_original':assert np.array_equal(arr,original)
    path=folder/'after.png';after.save(path)
    np.savez_compressed(folder/'masks_exact.npz',hair_lock=hair,eye_core=eye,alpha=alpha)
    Image.fromarray(hair.astype('uint8')*255).save(folder/'hair_rgb_lock.png')
    Image.fromarray(eye.astype('uint8')*255).save(folder/'eye_core.png')
    overlay=original.copy();overlay[eye]=(overlay[eye]*.4+np.array([255,0,0])*.6).astype('uint8')
    Image.fromarray(overlay).save(folder/'protection_overlay.png')
    row=dict(**{k:rec[k] for k in ['split','trial','case','branch','epoch']},path=str(path),before_path=str(before_path),
        sha256=sha(path),status=status,reason=reason,seconds=time.perf_counter()-seconds,
        hair_max_abs_RGB_change=int(np.abs(arr.astype(np.int16)-original.astype(np.int16))[hair].max()),
        eye_max_abs_RGB_change=int(np.abs(arr.astype(np.int16)-original.astype(np.int16))[eye].max()) if eye.any() else None)
    OUTCOMES.append(row);done[key_of(row)]=row;atomic_json(ledger,OUTCOMES)
    print(f'{len(done)}/{len(BASE_RECORDS)}',rec['split'],rec['trial'],rec['case'],rec['branch'],status,reason,flush=True)
assert set(done)=={key_of(r) for r in BASE_RECORDS} and len(OUTCOMES)==len(BASE_RECORDS)
pd.DataFrame(OUTCOMES).to_csv(OUT/'all_outcomes.csv',index=False)
pd.DataFrame(OUTCOMES).groupby(['split','trial','branch','status']).size().unstack(fill_value=0).to_csv(OUT/'outcome_counts.csv')
if hasattr(inpaint_pipe,'remove_all_hooks'):inpaint_pipe.remove_all_hooks()
inpaint_pipe.to('cpu');seg_model.cpu();gc.collect();torch.cuda.empty_cache()
id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)


In [ ]:
@torch.no_grad()
def image_tensor(path):
    with Image.open(path) as im:a=np.asarray(im.convert('RGB')).copy()
    return torch.from_numpy(a).float().permute(2,0,1)[None].cuda()/255.

REFERENCE_CACHE={};SOURCE_EYE_FAILURES=[]
def shared_reference(split,key):
    cache_key=(split,key)
    if cache_key not in REFERENCE_CACHE:
        folder=REPORT_ROOT/split/'images'/key
        d={name:image_tensor(folder/(name+'.png')).cpu() for name in ['face','ref']}
        for name in ['target','protected','eval_core','eval_refmask']:d[name]=read_mask(folder/(name+'.png'))
        assert d['face'].shape[-2:]==(256,256) and d['ref'].shape[-2:]==(256,256)
        with Image.open(folder/'face.png') as im:source=im.convert('RGB').copy()
        try:
            eye=build_eye_protection(source.size,detect_eye_landmarks(source))['eye_core']
        except ValueError as error:
            if not str(error).startswith('Eye protection:'):raise
            eye=np.zeros((source.height,source.width),dtype=bool)
            SOURCE_EYE_FAILURES.append(dict(split=split,case=key,status='source_eye_metric_unavailable',message=str(error)))
            pd.DataFrame(SOURCE_EYE_FAILURES).to_csv(Path(OUT)/'source_eye_metric_failures.csv',index=False)
        d['source_eye']=torch.from_numpy(eye.copy())[None,None]
        for name in ['eval_core','eval_refmask','protected']:assert d[name].any()
        REFERENCE_CACHE[cache_key]=d
    return gpu(REFERENCE_CACHE[cache_key])

@torch.no_grad()
def predicted_hair(im):
    labels=segment(align.to_bisenet(im))
    return F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5

@torch.no_grad()
def metrics_for(path,rec):
    full=image_tensor(path);im=to_size(full,256)
    d=shared_reference(rec['split'],rec['case'])
    L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
    vals=dict(L_quantile=float(L),ab_quantile=float(ab),
        identity_cosine=float((id_features(im)*id_features(d['face'])).sum(1).mean()),
        protected_RGB_MAE_vs_input=float(masked_mae(im,d['face'],d['protected'])),
        source_eye_RGB_MAE=float(masked_mae(im,d['face'],d['source_eye'])) if d['source_eye'].any() else float('nan'))
    vals.update(extra_metrics(im,d))
    vals.update({'proxy_'+k:v for k,v in shape_metrics(im,d).items()})
    pred=predicted_hair(im).cpu()
    assert all(np.isfinite(v) for k,v in vals.items() if k!='source_eye_RGB_MAE')
    return vals,pred



## เมทริกและกราฟครบทุกคู่
กราฟหลักเป็นผลของระบบรวม fallback ให้ดูอัตรา diffusion_success พร้อมกัน ไม่เปรียบว่า fallback เป็นความสำเร็จของ diffusion

In [ ]:
METRIC_ROWS=[]
for rec in BASE_RECORDS:
    for stage,path in [('before',rec['path']),('after',done[key_of(rec)]['path'])]:
        vals,_=metrics_for(path,rec)
        METRIC_ROWS.append(dict(**{k:rec[k] for k in ['split','trial','case','branch','epoch']},stage=stage,
            outcome='input' if stage=='before' else done[key_of(rec)]['status'],**vals))
        pd.DataFrame(METRIC_ROWS).to_csv(OUT/'metrics.partial.csv',index=False)
frame=pd.DataFrame(METRIC_ROWS)
assert len(frame)==2*len(BASE_RECORDS)
assert not frame.duplicated(['split','trial','case','branch','stage']).any()
frame.to_csv(OUT/'per_image_metrics.csv',index=False);(OUT/'metrics.partial.csv').unlink()
MAIN=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),('identity_cosine','Face identity cosine ↑'),
 ('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),('protected_SSIM','Protected-region SSIM ↑'),('protected_PSNR','Protected-region PSNR ↑')]
PROXY=[('proxy_target_mask_IoU','HFG-derived proxy IoU ↑'),('proxy_target_Dice','HFG-derived proxy Dice ↑'),('proxy_boundary_F1_2px','HFG-derived proxy boundary F1 ↑')]
metrics=[k for k,_ in MAIN+PROXY]+['source_eye_RGB_MAE']
means=frame.groupby(['split','trial','branch','stage'])[metrics].mean()
sds=frame.groupby(['split','trial','branch','stage'])[metrics].std(ddof=1)
means.to_csv(OUT/'mean_metrics_all_cases.csv');sds.to_csv(OUT/'sd_across_cases.csv')
frame.groupby(['split','trial','branch','stage'])[metrics].count().to_csv(OUT/'valid_case_count_per_metric.csv')
deltas=[]
for rec in BASE_RECORDS:
    subset=frame[(frame.split==rec['split'])&(frame.trial==rec['trial'])&(frame.case==rec['case'])&(frame.branch==rec['branch'])].set_index('stage')
    assert set(subset.index)=={'before','after'}
    deltas.append(dict(**{k:rec[k] for k in ['split','trial','case','branch']},status=done[key_of(rec)]['status'],
        **{m:float(subset.loc['after',m]-subset.loc['before',m]) for m in metrics}))
pd.DataFrame(deltas).to_csv(OUT/'paired_after_minus_before.csv',index=False)
for split,keys in SPLITS.items():
    for trial in TRIALS:
        for branch in BRANCHES:
            for stage in ['before','after']:
                part=frame[(frame.split==split)&(frame.trial==trial)&(frame.branch==branch)&(frame.stage==stage)]
                assert set(part.case)==set(keys) and len(part)==len(keys),'Missing case before final report'
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
COLORS=['#2875BA','#7E57C2','#D77A29']
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'figure.facecolor':'white'})
with PdfPages(OUT/'all_cases_report.pdf') as pdf:
    for split,keys in SPLITS.items():
        for trial in TRIALS:
            folder=OUT/'figures'/split/trial;folder.mkdir(parents=True,exist_ok=True)
            count_info=[]
            for branch in BRANCHES:
                records=[r for r in OUTCOMES if r['split']==split and r['trial']==trial and r['branch']==branch]
                count_info.append(f'{branch}: diffusion {sum(r["status"]=="diffusion_success" for r in records)}/{len(keys)}, fallback {sum(r["status"]=="fallback_original" for r in records)}')
            for group,specs in [('primary',MAIN),('shape_proxy',PROXY)]:
                fig,axes=plt.subplots(len(specs)//3,3,figsize=(16,4.8*(len(specs)//3)),squeeze=False)
                for ax,(metric,title) in zip(axes.flat,specs):
                    for j,branch in enumerate(BRANCHES):
                        ks=[(split,trial,branch,stage) for stage in ['before','after']]
                        ax.errorbar([j*3,j*3+1],[means.loc[k,metric] for k in ks],yerr=[sds.loc[k,metric] for k in ks],fmt='o-',color=COLORS[j],capsize=3)
                    ax.set_xticks([0,1,3,4,6,7],['ON\nbefore','ON\nafter','OFF\nbefore','OFF\nafter','BS\nbefore','BS\nafter'])
                    ax.set_title(title,loc='left');ax.grid(alpha=.2);ax.spines[['top','right']].set_visible(False)
                fig.suptitle(f'{split} | {trial} | ALL {len(keys)} cases, mean ± SD | Includes labeled fallback outputs')
                fig.text(.03,.015,' | '.join(count_info),fontsize=8)
                fig.tight_layout(rect=(0,.055,1,.94));fig.savefig(folder/(group+'.png'),dpi=170);pdf.savefig(fig);plt.close(fig)
            for key in keys:
                fig,axes=plt.subplots(3,2,figsize=(10,13))
                for i,branch in enumerate(BRANCHES):
                    rec=done[(split,trial,key,branch)]
                    for j,(path,label) in enumerate([(rec['before_path'],'Before'),(rec['path'],rec['status'])]):
                        with Image.open(path) as im:axes[i,j].imshow(im)
                        axes[i,j].set_title(branch+' | '+label,fontsize=10);axes[i,j].axis('off')
                fig.suptitle(f'{split}/{key} | {trial}');fig.tight_layout()
                fig.savefig(folder/(key+'.png'),dpi=130);pdf.savefig(fig);plt.close(fig)
for path,digest in INPUT_HASHES.items():assert sha(Path(path))==digest
manifest=dict(status='complete',scheduled_cases={s:len(k) for s,k in SPLITS.items()},expected_after=len(BASE_RECORDS),actual_after=len(OUTCOMES),
    actual_metric_rows=len(frame),diffusion_success=sum(r['status']=='diffusion_success' for r in OUTCOMES),
    background_only=sum(r['status']=='background_only' for r in OUTCOMES),fallback_original=sum(r['status']=='fallback_original' for r in OUTCOMES),
    no_training=True,no_cases_excluded=True,source_images_unchanged=True,checkpoint_lock_from_source=SOURCE_LOCK,
    evaluation='end-to-end pipeline with explicit fallback, not a claim of universal diffusion success',
    limitations=['hair RGB equality only inside locked segmentation/target mask','shape target is HFG-derived proxy',
                 'background sharpening is not ground-truth detail recovery','previously viewed test; no tuning or selection from test',
                 'source-eye metric can be unavailable independently of other metrics'])
(OUT/'completion.json').write_text(json.dumps(manifest,indent=2))
(OUT/'README_TH.txt').write_text('''รายงานครบทุกคู่โดยไม่ตัดตัวอย่าง
ค่าเฉลี่ยรวมผลสำเร็จและ fallback ของระบบ; ไม่อ้างว่าทุกภาพผ่าน diffusion สำเร็จ
fallback_original = คืนภาพก่อน diffusion เหมือนเดิมทุกพิกเซล ไม่มีการเดา mask ตา
background_only = ไม่มีพื้นที่ diffusion เหลือ ปรับเฉพาะพื้นหลัง
ดู all_outcomes.csv และ outcome_counts.csv ควบคู่กับคะแนนเสมอ
ผมล็อก RGB ภายใน mask: ตรวจ hair_rgb_lock.png ให้ครอบผมครบ
พื้นหลัง sharpen อาจทำให้ MAE/SSIM/PSNR เทียบต้นฉบับแย่ลง; คะแนนเหล่านี้วัดการรักษาภาพ ไม่ใช่ความชัดที่ถูกต้อง
ไม่มีการฝึกใหม่หรือเลือก checkpoint/prompt/seed จาก test
''',encoding='utf-8')
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
display(means);display(pd.DataFrame(OUTCOMES).groupby(['split','trial','branch','status']).size())
print('COMPLETE:',manifest,'\nZIP:',archive)
files.download(archive)
